# 03 — ML Frame Scoring (Optional / Supplementary)

Requires: `pip install -r requirements-ml.txt` (then install PyTorch for your hardware — see that file).

Two approaches that complement the dictionary-based frame flags:

1. **Embedding scoring** (FrameAxis-inspired, Kwak et al. 2021) — sentence-transformer cosine similarity, fast, multilingual.  
2. **Zero-shot NLI** (Laurer et al. 2024) — mDeBERTa entailment probabilities, slower, no labelled data needed.

Both are run on a stratified sample of the corpus (see `SAMPLE_N`).  
Results are saved to `data/processed/` and optionally rendered as supplementary figures in `02_paper_figures.ipynb`.

| Section | What it does | Output |
|---------|-------------|--------|
| 1 | Load parquet (text + metadata only) + stratified sample | — |
| 2 | Embedding scoring | `ml_frame_scores_embedding.parquet` |
| 3 | NLI scoring + build comparison dataset | `ml_frame_scores_nli.parquet`, `ml_comparison.parquet` |
| 4a | Dictionary validation (t-tests, Cohen's κ) | printed table |
| 4b | Embedding vs. NLI comparison (Spearman ρ) | printed table |

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dictionaries import FRAME_COLS, FRAME_DICTS

INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"

PREPROCESSED        = INTERIM  / "gdelt_preprocessed.parquet"
ML_EMBEDDING_PATH   = PROCESSED / "ml_frame_scores_embedding.parquet"
ML_NLI_PATH         = PROCESSED / "ml_frame_scores_nli.parquet"
ML_COMPARISON_PATH  = PROCESSED / "ml_comparison.parquet"

FRAME_NAMES = [c.replace("frame_", "") for c in FRAME_COLS]

SAMPLE_N    = 1_000   # None = full corpus (~1.5 h on CPU)
NLI_N       = 500    # NLI is ~2-5 s/article on CPU — keep small
RANDOM_SEED = 42

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("No GPU detected — running on CPU (NLI will be slow)")
except ImportError:
    DEVICE = "cpu"
    print("torch not installed — run: pip install -r requirements-ml.txt")

GPU: AMD Radeon RX 9070 XT


## Section 1 — Load text (memory-efficient)

Loads only `Quotations` + frame metadata columns from the preprocessed parquet.
This is ~20× smaller than loading the full 2 GB parquet.

In [2]:
NEEDED_COLS = ["Quotations", "dominant_frame", "month", "region"] + FRAME_COLS
df_meta = pd.read_parquet(PREPROCESSED, columns=NEEDED_COLS)
mem_mb = df_meta.memory_usage(deep=True).sum() / 1e6
print(f"Loaded {len(df_meta):,} articles  ({mem_mb:.0f} MB in memory)")

if SAMPLE_N is not None and SAMPLE_N < len(df_meta):
    n_months = df_meta["month"].nunique()
    sample_per_month = max(1, SAMPLE_N // n_months)
    # Groupby on a string proxy to avoid pandas dropping Period[M] columns in apply/apply chains.
    # Each `g` slice inherits all columns from df_meta including the Period `month`.
    ym_str = df_meta["month"].astype(str)
    df_sample = pd.concat(
        [g.sample(min(len(g), sample_per_month), random_state=RANDOM_SEED)
         for _, g in df_meta.groupby(ym_str, observed=True)],
        ignore_index=True,
    )
    if len(df_sample) > SAMPLE_N:
        df_sample = df_sample.sample(SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)
else:
    df_sample = df_meta.reset_index(drop=True)

texts = df_sample["Quotations"].fillna("").tolist()
print(f"Sample: {len(df_sample):,} articles across {df_sample['month'].nunique()} months")

Loaded 1,116,091 articles  (209 MB in memory)
Sample: 968 articles across 44 months


## Section 2 — Embedding scoring

FrameAxis-inspired cosine similarity between article texts and frame keyword centroids.
Scores are in [−1, 1]; higher = semantically closer to the frame.

Skipped if `ml_frame_scores_embedding.parquet` already exists.

In [3]:
from src.framing_scores import assign_frame_scores_embedding

if ML_EMBEDDING_PATH.exists():
    scores_emb = pd.read_parquet(ML_EMBEDDING_PATH)
    print(f"Loaded from cache: {ML_EMBEDDING_PATH.name}  ({len(scores_emb):,} rows)")
else:
    print(f"Scoring {len(texts):,} articles (model loads once, GPU batches internally)...")
    emb_scores = assign_frame_scores_embedding(texts, FRAME_DICTS)
    meta_cols = ["dominant_frame", "month", "region"] + FRAME_COLS
    scores_emb = pd.concat(
        [df_sample[meta_cols].reset_index(drop=True), emb_scores],
        axis=1,
    )
    scores_emb.to_parquet(ML_EMBEDDING_PATH, index=False)
    print(f"Saved {len(scores_emb):,} rows → {ML_EMBEDDING_PATH}")

Scoring 968 articles (model loads once, GPU batches internally)...


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Saved 968 rows → /home/Brewen/cwd/tud/GenAI-GDELT/data/processed/ml_frame_scores_embedding.parquet


## Section 3 — NLI scoring + comparison dataset

Zero-shot mDeBERTa classification on the **same first `NLI_N` articles** scored in Section 2,
so the two approaches are directly comparable on identical data.

The comparison dataset (`ml_comparison.parquet`) merges both sets of scores and adds
`dominant_frame_emb` and `dominant_frame_nli` columns (argmax of each method).

Skipped if `ml_frame_scores_nli.parquet` already exists.

In [4]:
from src.framing_scores import assign_frame_scores_nli

nli_subset = df_sample.iloc[:NLI_N].copy().reset_index(drop=True)
nli_texts  = nli_subset["Quotations"].fillna("").tolist()

if ML_NLI_PATH.exists():
    scores_nli = pd.read_parquet(ML_NLI_PATH)
    print(f"Loaded from cache: {ML_NLI_PATH.name}  ({len(scores_nli):,} rows)")
else:
    print(f"Scoring {NLI_N} articles with NLI classifier (first call downloads ~700 MB model)...")
    nli_raw = assign_frame_scores_nli(nli_texts)
    nli_raw = nli_raw.rename(columns={n: f"nli_{n}" for n in FRAME_NAMES})
    meta_cols = ["dominant_frame", "month", "region"] + FRAME_COLS
    scores_nli = pd.concat(
        [nli_subset[meta_cols], nli_raw.reset_index(drop=True)],
        axis=1,
    )
    scores_nli.to_parquet(ML_NLI_PATH, index=False)
    print(f"Saved {len(scores_nli):,} rows → {ML_NLI_PATH}")

# Build merged comparison dataset on the shared NLI_N rows
emb_nli_rows = scores_emb.iloc[:NLI_N].reset_index(drop=True)
nli_score_cols = [f"nli_{n}" for n in FRAME_NAMES]
comparison = pd.concat([
    emb_nli_rows,
    scores_nli[nli_score_cols].reset_index(drop=True),
], axis=1)

# Dominant frame per ML method (argmax)
comparison["dominant_frame_emb"] = (
    emb_nli_rows[FRAME_NAMES].idxmax(axis=1)
)
comparison["dominant_frame_nli"] = (
    scores_nli[nli_score_cols]
    .rename(columns={f"nli_{n}": n for n in FRAME_NAMES})
    .idxmax(axis=1)
    .values
)

comparison.to_parquet(ML_COMPARISON_PATH, index=False)
print(f"Comparison dataset: {len(comparison):,} articles → {ML_COMPARISON_PATH}")

Scoring 500 articles with NLI classifier (first call downloads ~700 MB model)...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Saved 500 rows → /home/Brewen/cwd/tud/GenAI-GDELT/data/processed/ml_frame_scores_nli.parquet
Comparison dataset: 500 articles → /home/Brewen/cwd/tud/GenAI-GDELT/data/processed/ml_comparison.parquet


## Section 4a — Dictionary validation

For each frame: mean embedding score and NLI probability for articles where the
keyword flag fires vs. where it does not. Welch's t-test.

**Expected:** both ML methods should score higher when the keyword flag fires,
validating the dictionary-based approach.

Also reports Cohen's κ for pairwise agreement on dominant frame assignment
(keyword vs. embedding vs. NLI).

In [6]:
from scipy import stats
from sklearn.metrics import cohen_kappa_score

print("Dictionary validation: mean ML score for keyword-matched vs. unmatched articles\n")
cols = f"{'Frame':<35} {'emb_hit':>8} {'emb_no':>8} {'emb_p':>10}  {'nli_hit':>8} {'nli_no':>8} {'nli_p':>10}"
print(cols)
print("-" * len(cols))

for name in FRAME_NAMES:
    col_kw = f"frame_{name}"
    if col_kw not in comparison.columns:
        continue
    has_flag = comparison[col_kw] > 0
    n_hit, n_no = has_flag.sum(), (~has_flag).sum()
    if n_hit < 2 or n_no < 2:
        print(f"{name:<35}  (insufficient data: {n_hit} matches)")
        continue

    emb_hit = comparison.loc[has_flag,  name].mean()
    emb_no  = comparison.loc[~has_flag, name].mean()
    nli_hit = comparison.loc[has_flag,  f"nli_{name}"].mean()
    nli_no  = comparison.loc[~has_flag, f"nli_{name}"].mean()

    _, p_emb = stats.ttest_ind(
        comparison.loc[has_flag,  name], comparison.loc[~has_flag, name], equal_var=False)
    _, p_nli = stats.ttest_ind(
        comparison.loc[has_flag,  f"nli_{name}"], comparison.loc[~has_flag, f"nli_{name}"], equal_var=False)

    print(f"{name:<35} {emb_hit:8.3f} {emb_no:8.3f} {p_emb:10.2e}  {nli_hit:8.3f} {nli_no:8.3f} {p_nli:10.2e}")

print()
valid = comparison["dominant_frame"].notna()
kw_arr  = comparison.loc[valid, "dominant_frame"].values
emb_arr = comparison.loc[valid, "dominant_frame_emb"].values
nli_arr = comparison.loc[valid, "dominant_frame_nli"].values

print(f"Cohen's κ on dominant frame  (n={valid.sum():,} articles with at least one keyword hit)")
try:
    print(f"  Keyword  vs. Embedding : κ = {cohen_kappa_score(kw_arr, emb_arr):.3f}")
    print(f"  Keyword  vs. NLI       : κ = {cohen_kappa_score(kw_arr, nli_arr):.3f}")
    print(f"  Embedding vs. NLI      : κ = {cohen_kappa_score(emb_arr, nli_arr):.3f}")
except Exception as e:
    print(f"  Failed: {e}")

Dictionary validation: mean ML score for keyword-matched vs. unmatched articles

Frame                                emb_hit   emb_no      emb_p   nli_hit   nli_no      nli_p
----------------------------------------------------------------------------------------------
innovation_opportunity                 0.189    0.180   5.32e-01     0.764    0.736   1.85e-01
risk_safety                            0.190    0.176   2.58e-01     0.456    0.469   6.42e-01
regulation_governance                  0.159    0.161   9.03e-01     0.470    0.428   1.05e-01
rights_privacy                         0.162    0.180   3.13e-01     0.404    0.399   8.94e-01
economic_competition_labour            0.186    0.175   4.35e-01     0.469    0.465   8.53e-01
misinformation_integrity               0.179    0.175   8.05e-01     0.882    0.920   1.61e-01

Cohen's κ on dominant frame  (n=246 articles with at least one keyword hit)
  Keyword  vs. Embedding : κ = 0.172
  Keyword  vs. NLI       : κ = 0.017
  Embedd

## Section 4b — Embedding vs. NLI head-to-head

Spearman ρ between embedding scores and NLI probabilities for each frame (diagonal = same frame,
off-diagonal = cross-frame). The diagonal values tell you how much the two methods agree
on which articles are relevant to each frame.

In [7]:
from scipy.stats import spearmanr

print("Spearman ρ: Embedding score vs. NLI probability (per frame)\n")
print(f"{'Frame':<35} {'ρ':>8} {'p':>12}  sig")
print("-" * 62)

for name in FRAME_NAMES:
    rho, p = spearmanr(comparison[name], comparison[f"nli_{name}"])
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"{name:<35} {rho:8.3f} {p:12.2e}  {sig}")

print("\nCross-frame Spearman ρ  (rows = embedding dimension, cols = NLI dimension)\n")
matrix = pd.DataFrame(
    [[round(spearmanr(comparison[r], comparison[f"nli_{c}"])[0], 3) for c in FRAME_NAMES]
     for r in FRAME_NAMES],
    index=FRAME_NAMES,
    columns=FRAME_NAMES,
)
print(matrix.to_string())

Spearman ρ: Embedding score vs. NLI probability (per frame)

Frame                                      ρ            p  sig
--------------------------------------------------------------
innovation_opportunity                 0.280     1.75e-10  ***
risk_safety                            0.337     1.02e-14  ***
regulation_governance                  0.359     1.21e-16  ***
rights_privacy                         0.287     5.73e-11  ***
economic_competition_labour            0.265     1.79e-09  ***
misinformation_integrity               0.267     1.37e-09  ***

Cross-frame Spearman ρ  (rows = embedding dimension, cols = NLI dimension)

                             innovation_opportunity  risk_safety  regulation_governance  rights_privacy  economic_competition_labour  misinformation_integrity
innovation_opportunity                        0.280        0.315                  0.303           0.273                        0.278                     0.259
risk_safety                             